**Official documentation:** [LangGraph Graph API Overview](https://docs.langchain.com/oss/python/langgraph/graph-api?utm_source=chatgpt.com)

---

# LangGraph Graph API – Complete Study Notes

## What is the Graph API?

The Graph API is the **core programming model** of LangGraph.

Everything in LangGraph is represented as a **graph** made of:

* **State** → Stores data
* **Nodes** → Perform work
* **Edges** → Decide where to go next

Think of it like this:

```text
State
   │
   ▼
Node A
   │
   ▼
Node B
   │
   ▼
Node C
```

Instead of executing code line by line, LangGraph moves through connected nodes while updating shared state. ([Docs by LangChain][1])

---

# The Three Core Components

## 1. State

State is the **shared memory** of your graph.

Every node:

* Reads the state
* Updates the state
* Passes the updated state to the next node

Example:

```python
from typing_extensions import TypedDict

class State(TypedDict):
    question: str
    answer: str
```

Example state during execution:

```python
{
    "question": "What is AI?",
    "answer": ""
}
```

After an LLM node:

```python
{
    "question": "What is AI?",
    "answer": "Artificial Intelligence..."
}
```

### Key Points

* Shared across all nodes
* Updated after every node
* Acts as the graph's memory
* Can use:

  * `TypedDict` (recommended)
  * `dataclass`
  * `Pydantic BaseModel` (validation, slower) ([Docs by LangChain][1])

---

## 2. Nodes

Nodes are **Python functions** that perform work.

A node can:

* Call an LLM
* Use tools
* Query a database
* Run Python logic
* Modify state

Example:

```python
def chatbot(state):
    return {
        "answer": "Hello!"
    }
```

Execution flow:

```text
State
   │
   ▼
Node executes
   │
   ▼
Updated State
```

### Node Inputs

A node may receive:

```python
def my_node(state, config, runtime):
    ...
```

* `state` → Current graph state
* `config` → Runtime configuration (thread ID, tracing, etc.)
* `runtime` → Context, stores, streaming, execution info, control signals ([Docs by LangChain][1])

---

## 3. Edges

Edges determine **what node runs next**.

Example:

```text
Node A
   │
   ▼
Node B
```

Code:

```python
graph.add_edge("A", "B")
```

This means:

```
Run A

↓

Run B
```

---

# Graph Execution Flow

The overall workflow is:

```text
Create State

↓

Create Graph

↓

Add Nodes

↓

Add Edges

↓

Compile

↓

Invoke
```

This is the standard lifecycle of every LangGraph application. ([Docs by LangChain][1])

---

# StateGraph

`StateGraph` is the main class used to build a graph.

```python
from langgraph.graph import StateGraph

graph = StateGraph(State)
```

Responsibilities:

* Holds the graph
* Tracks state schema
* Stores nodes
* Stores edges

---

# Building a Graph

### Step 1

Create state

```python
class State(TypedDict):
    message: str
```

---

### Step 2

Create graph

```python
builder = StateGraph(State)
```

---

### Step 3

Add node

```python
builder.add_node("chatbot", chatbot)
```

---

### Step 4

Connect nodes

```python
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
```

---

### Step 5

Compile

```python
graph = builder.compile()
```

---

### Step 6

Run

```python
graph.invoke(...)
```

---

# START Node

`START` is a virtual node representing the beginning of execution.

```python
builder.add_edge(
    START,
    "chatbot"
)
```

Execution:

```text
START

↓

chatbot
```

---

# END Node

`END` is the virtual node where execution finishes.

```python
builder.add_edge(
    "chatbot",
    END
)
```

Execution:

```text
chatbot

↓

END
```

---

# Compile

A graph **must** be compiled before it can run.

```python
graph = builder.compile()
```

Compilation:

* Validates the graph
* Checks for orphaned nodes
* Configures runtime features (for example, checkpointers and breakpoints)

Without `compile()`, `invoke()` will not work. ([Docs by LangChain][1])

---

# Conditional Edges

Instead of always going to the same node, you can route based on logic.

```python
builder.add_conditional_edges(
    "agent",
    routing_function
)
```

Example:

```text
Agent

↓

Need Tool?

Yes → Tool

No → END
```

This enables loops and decision-making. ([Docs by LangChain][1])

---

# Conditional Entry Point

You can even decide the **first** node dynamically.

```python
builder.add_conditional_edges(
    START,
    router
)
```

Example:

```text
START

↓

Question?

↓

Math Node

or

Chat Node
```

---

# Command

`Command` lets a node **update state and decide the next node** in one return value.

Example:

```python
return Command(
    update={...},
    goto="tool_node"
)
```

It supports:

* `update` → modify state
* `goto` → choose next node
* `graph` → navigate parent graphs
* `resume` → continue after an interrupt

This can replace separate conditional edges in many workflows. ([Docs by LangChain][1])

---

# Runtime Context

Runtime context passes information that **isn't part of the graph state**.

Example:

```python
context = {
    "llm_provider": "openai"
}
```

Nodes access it through `runtime.context`.

Use it for:

* Model selection
* Database connections
* API clients
* Configuration

This keeps transient dependencies separate from persistent state. ([Docs by LangChain][1])

---

# Graph Execution Model (Super-Steps)

LangGraph is inspired by **Google Pregel**.

Execution occurs in **super-steps**:

1. Active nodes execute.
2. They send updated state to connected nodes.
3. Newly activated nodes execute in the next super-step.
4. The graph ends when no nodes remain active.

Conceptually:

```text
Step 1

START
 │
 ▼
Node A

↓

Step 2

Node B
Node C

↓

Step 3

END
```

Parallel nodes run in the same super-step. ([Docs by LangChain][1])

---

# Recursion Limit

To avoid infinite loops, LangGraph limits the number of super-steps.

```python
graph.invoke(
    inputs,
    {
        "recursion_limit": 20
    }
)
```

If the limit is exceeded, it raises `GraphRecursionError`.

You can also monitor remaining steps with the managed value `RemainingSteps` for graceful exits. ([Docs by LangChain][1])

---

# Graph Migration

LangGraph supports evolving graphs over time.

Supported changes include:

* Adding nodes
* Removing nodes (with restrictions for interrupted runs)
* Changing edges
* Adding or removing state keys

This is useful when using persistent checkpoints. ([Docs by LangChain][1])

---

# Complete Flow Diagram

```text
User Input
     │
     ▼
START
     │
     ▼
Node 1
     │
     ▼
Node 2
     │
     ▼
Decision?
 ┌───────────────┐
 │               │
 ▼               ▼
Tool Node       END
 │
 ▼
Node 2
```

---

# Important APIs

| API                       | Purpose                         |
| ------------------------- | ------------------------------- |
| `StateGraph()`            | Create the graph                |
| `add_node()`              | Add a node                      |
| `add_edge()`              | Connect nodes                   |
| `add_conditional_edges()` | Dynamic routing                 |
| `compile()`               | Validate and prepare the graph  |
| `invoke()`                | Execute the graph               |
| `START`                   | Entry point                     |
| `END`                     | Exit point                      |
| `Command`                 | Update state and route together |

---

# Interview Questions

1. What is the Graph API?
2. What are the three main components of a LangGraph?
3. What is the purpose of `StateGraph`?
4. Why is `compile()` required?
5. What is shared state?
6. How do nodes communicate?
7. What are conditional edges?
8. What is the difference between `START` and `END`?
9. What is `Command` and when would you use it?
10. What are super-steps?
11. What is runtime context?
12. What is the recursion limit?

---

# Cheat Sheet

```text
Graph API

State
  ↓
Node
  ↓
Edge
  ↓
Next Node

Main APIs
---------
StateGraph()
add_node()
add_edge()
add_conditional_edges()
compile()
invoke()

Special Nodes
-------------
START
END

Advanced
--------
Command
Runtime Context
Conditional Routing
Recursion Limit
Graph Migration
```

## Recommended next topics

To build practical LangGraph applications, study these in order:

1. **Graph API** ✅
2. State
3. Nodes
4. Edges
5. Reducers
6. MessagesState
7. Checkpointers
8. Memory
9. Human-in-the-Loop
10. Multi-Agent Systems